<a href="https://colab.research.google.com/github/Shashini294/Statistical-Learning-e22294/blob/main/Assignment_7c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## Part 1: Bayesian Estimation of a User Ability Parameter from Item Responses

### 1. Visualizing the Mechanics & Curve Interpretation

* **Interpretation of Moving $b_i$:** The difficulty parameter $b_i$ represents the value of $\theta$ at which the probability of a correct response is $0.5$ ($P(Y_i = 1 \mid \Theta = b_i) = 0.5$).
* Increasing $b_i$ (making the item harder) shifts the item response curve **horizontally to the right**, requiring a higher ability $\theta$ to achieve the same probability of success.
* Decreasing $b_i$ (making the item easier) shifts the curve **horizontally to the left**.



---

### 2. Sequential Likelihood & Joint Likelihood

* **Single Observation Likelihood Contribution:**

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$



where $p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$.
* **Joint Likelihood for History Vector $y^{(k)} = (y_1, y_2, \dots, y_k)$:**
Assuming conditional independence given $\theta$:

$$L(y^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$



---

### 3. Mathematical Formulation of the Running Update

The recursive relationship for the posterior density at step $k$ up to a proportionality constant is:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

Explicitly written with the normalizing constant:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) = \frac{[p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})}{\int_{-\infty}^{\infty} [p_k(s)]^{y_k} [1 - p_k(s)]^{1 - y_k} \cdot f_{\Theta \mid Y^{(k-1)}}(s \mid y^{(k-1)}) \, ds}$$

---

### 4. Dynamic Shifting

When a user correctly answers ($y_k = 1$) a highly difficult item (large $b_k$), $p_k(\theta)$ is very low for average or lower ability values, but increases significantly for higher $\theta$ values. Multiplying the prior density $f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$ by this monotonically increasing likelihood function suppresses low $\theta$ values and scales up high $\theta$ values. Consequently, the mass and peak (mode) of the posterior density shift **significantly to the right**, indicating a substantial upward revision of estimated user ability.

---

### 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls the slope (steepness) of the item response function at $\theta = b_k$:

* **Very Large $a_k$:** Provides high information around $b_k$. The likelihood function transitions sharply from near $0$ to near $1$. This causes a rapid narrowing of the posterior distribution (**higher sharpness / reduced variance**), signaling high precision.
* **Very Small $a_k$:** The likelihood is relatively flat across $\theta$, offering minimal information about ability. Updating with a low $a_k$ item barely changes the prior, keeping the posterior variance wide.

---

### 6. Numerical Implementation on a Fixed Grid

Since the 2PL model does not yield a conjugate prior, updates are evaluated over a discrete grid $\Theta_{\text{grid}} = \{\theta_1, \theta_2, \dots, \theta_M\}$:

1. **Initialize:** Set $f^{(0)}(\theta_m) = \text{NormPDF}(\theta_m, 0, 1)$ across all grid points.
2. **Sequential Likelihood Evaluation:** For step $k$, compute $L(y_k \mid \theta_m) = [p_k(\theta_m)]^{y_k} [1 - p_k(\theta_m)]^{1 - y_k}$.
3. **Unnormalized Update:** Compute $f_{\text{unnorm}}(\theta_m) = L(y_k \mid \theta_m) \times f^{(k-1)}(\theta_m)$.
4. **Sequential Normalization Step:** Numerically approximate the continuous integral using the trapezoidal rule:

$$\text{Integral} = \sum_{m=1}^{M-1} \frac{f_{\text{unnorm}}(\theta_m) + f_{\text{unnorm}}(\theta_{m+1})}{2} \Delta \theta$$



Set $f^{(k)}(\theta_m) = \frac{f_{\text{unnorm}}(\theta_m)}{\text{Integral}}$ so that $\int f^{(k)}(\theta) \, d\theta \approx 1$.

---

### 7. Python Implementation (Convergence Timeline)

```python
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set seed for reproducibility
np.random.seed(42)

# 1. Setup Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

# 2. Random Item Parameters
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Trackers
running_bayes = [0.0]  # Step 0 prior mean = 0
running_map = [0.0]    # Step 0 prior MAP = 0
steps = list(range(n_items + 1))

# Initialize Prior N(0, 1)
current_posterior = stats.norm.pdf(theta_grid, 0, 1)

# Simulation Loop
for k in range(n_items):
    a_k, b_k = a_params[k], b_params[k]
    
    # Generate stochastic user response
    p_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0
    
    # Calculate likelihood across grid
    p_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))
    
    # Update and normalize
    current_posterior *= likelihood
    integral = np.trapezoid(current_posterior, theta_grid)
    current_posterior /= integral
    
    # Point estimates
    theta_bayes = np.trapezoid(theta_grid * current_posterior, theta_grid)
    theta_map = theta_grid[np.argmax(current_posterior)]
    
    running_bayes.append(theta_bayes)
    running_map.append(theta_map)

# Visualization
fig = go.Figure()
fig.add_hline(y=theta_true, line_dash="dash", line_color="red",
              annotation_text=f"True Ability (θ = {theta_true})")

fig.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines+markers',
                         name='Posterior Mean (θ̂_Bayes)', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=steps, y=running_map, mode='lines+markers',
                         name='MAP Estimate (θ̂_MAP)', line=dict(color='green', dash='dot')))

fig.update_layout(
    title="Convergence of Latent Ability Estimators (θ) Over Time",
    xaxis_title="Item Sequence (k)",
    yaxis_title="Estimated Ability (θ̂)",
    template="plotly_white"
)
fig.show()

```

* **Convergence Analysis:** As $k$ increases, additional observations narrow the posterior density variance. Both estimators ($\hat{\theta}_{\text{Bayes}}$ and $\hat{\theta}_{\text{MAP}}$) move closer to $\theta_{\text{true}} = 0.75$, reflecting the system's growing certainty about the user's skill level as data accumulates.

---

---

## Part 2: Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

### 1. Structural Probability and Density Interpretation

* **Beta Distribution Dynamics:**
* $\alpha = \beta$: Symmetric distribution centered at $\theta = 0.5$.
* $\alpha < \beta$ (Right-Skewed): Concentration of mass towards $0$, representing low CTR beliefs.
* $\alpha > \beta$ (Left-Skewed): Concentration of mass towards $1$, representing high CTR beliefs.
* Larger values of $\alpha + \beta$ increase sharpness (reduce variance), reflecting stronger prior weight.



---

### 2. Sequential Likelihood and Joint History

* **Single Impression Likelihood:**

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$


* **Joint Likelihood Vector $y^{(k)}$:**

$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum y_i} (1 - \theta)^{k - \sum y_i}$$



---

### 3. Closed-Form Analytical Updates (Beta-Binomial Conjugacy Proof)

Given step $k-1$ posterior $f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \sim \text{Beta}(\alpha_{k-1}, \beta_{k-1})$:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This retains the functional form of a Beta distribution, proving conjugacy.

* **Parameter Update Rules:**

$$\alpha_k = \alpha_{k-1} + y_k$$


$$\beta_k = \beta_{k-1} + (1 - y_k)$$


* **Posterior Mean:**

$$\mathbb{E}[\Theta \mid y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$



---

### 4. Dynamic Shifting Mechanics

* **Update Effect:** An observed click ($y_k = 1$) increments $\alpha$ by 1 while leaving $\beta$ unchanged, increasing the numerator and shifting the distribution peak to the right. A non-click ($y_k = 0$) increments $\beta$ by 1, shifting the peak left.
* **Analytical vs. Grid Integration:** Conjugacy permits $O(1)$ arithmetic parameter updates at each step, avoiding grid approximations or numerical quadrature required in models like the 2PL IRT model.

---

### 5. Running Point Estimators (Closed-Form Formulas)

* **Running Posterior Mean ($\hat{\theta}^{(k)}_{\text{Bayes}}$):**

$$\hat{\theta}^{(k)}_{\text{Bayes}} = \frac{\alpha_k}{\alpha_k + \beta_k}$$


* **Running MAP Estimate ($\hat{\theta}^{(k)}_{\text{MAP}}$):**

$$\hat{\theta}^{(k)}_{\text{MAP}} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad \text{for } \alpha_k, \beta_k > 1$$



---

### 6. Performance Tracking and Convergence Analysis (Python Code)

```python
import numpy as np
import plotly.graph_objects as go

# 1. Setup Simulation
np.random.seed(42)
theta_true = 0.35
n_impressions = 100

# Base Uninformative Prior Beta(1, 1)
alpha_k = 1
beta_k = 1

running_bayes = [alpha_k / (alpha_k + beta_k)]
running_map = [(alpha_k - 1) / (alpha_k + beta_k - 2)]  # 0.5 under Beta(1,1)
steps = list(range(n_impressions + 1))

# 2. Simulation Loop
for k in range(1, n_impressions + 1):
    # Draw stochastic click event
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    
    # Closed-form analytical updates
    alpha_k += y_k
    beta_k += (1 - y_k)
    
    # Store point estimates
    mean_est = alpha_k / (alpha_k + beta_k)
    map_est = (alpha_k - 1) / (alpha_k + beta_k - 2)
    
    running_bayes.append(mean_est)
    running_map.append(map_est)

# 3. Visualization
fig = go.Figure()
fig.add_hline(y=theta_true, line_dash="dash", line_color="red",
              annotation_text=f"True CTR (θ = {theta_true})")

fig.add_trace(go.Scatter(x=steps, y=running_bayes, mode='lines',
                         name='Posterior Mean (θ̂_Bayes)', line=dict(color='blue', width=2)))
fig.add_trace(go.Scatter(x=steps, y=running_map, mode='lines',
                         name='MAP Estimate (θ̂_MAP)', line=dict(color='green', dash='dot')))

fig.update_layout(
    title="Beta-Binomial Sequential CTR Estimation Convergence",
    xaxis_title="Impression Step (k)",
    yaxis_title="Estimated CTR (θ̂)",
    template="plotly_white"
)
fig.show()

```

* **Analysis:** As sample size $k \to 100$, accumulated observational evidence dominates the prior belief $\text{Beta}(1,1)$. Both estimates stabilize and narrow around $\theta_{\text{true}} = 0.35$, reducing estimator variance according to standard asymptotic properties ($Var(\theta) \propto \frac{1}{k}$).